# Marks Calculator - ECON2102 (2026, S1) HQ + WS

by [MachinaFantasma](https://phantomachine.github.io/)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import scipy.interpolate as interp

## Specify file path(s) and names

In [2]:
! mkdir official/

mkdir: official/: File exists


In [3]:
# Source file
outfile = "ECON2102-HQ+WS"

# Target output file
file_out = outfile + ".xlsx"
stats_out = outfile + "-stats" + ".xlsx"

# Official admin sheet
officialpath = "official/"
# officialfile = officialpath + "ECON2102.xlsx"
official_outfile = officialpath + "ECON2102-hurdles-out.csv"

# What stats would you like to report?
stat_au_choix = [np.mean, np.median, np.std, min, max, len]

## Import from CANVAS marks

In [4]:
# Import as dataframe
df = pd.read_csv("2026-06-06-ECON2102-from-canvas.csv")

## All activity keys

* Assessed and non-assessed activities

* Use this to view and decide which keys are applicable to hurdle assessement items

* Note: This also tells you how much **ongoing self-test and learning activities with feedback** exist throughout this course.

In [5]:
df.keys()

Index(['LastName', 'FirstName', 'ID', 'SIS User ID', 'SIS Login ID',
       'Integration ID', 'Section',
       'PT01 Leotards and Legwarmers Session (33864)',
       'WQ02-01 Micro v Macro (PS1-Q2) (33392)',
       'WQ02-02 Growth Rates (33389)', 'WQ02-00 Output Gap (33333)',
       'PT02 Measurement + Micro Warmups (33862)',
       'WQ03-01 GDP per person - Pros and Cons (33834)',
       'WQ03-02 Accounting - Measuring GDP items (33837)',
       'PT03 More Micro Warmups (33866)',
       'WQ04-01 Consumer Preferences (properties) (34473)',
       'WQ04-02 Preference Representation - ICs (34474)',
       'WQ04-03 Two-period Model: Walras (34475)',
       'PT04+HQ01 Static General Equilibrium (Hurdle Assessment) (33863)',
       'WQ05-01 Firms and technology (34982)',
       'PT05 One-period General Equilibrium + Applications (33865)',
       'WQ06-01 Competitive Equilibrium (Static) (35456)',
       'WQ06-02 Static GE + distortionary labor income tax (35470)',
       'HQ02 (Week 6) Sta

## Define selections of `keys`

Get from `df.keys()` above. Create keys for:
* `CANVAS` student identifiers and attributes.
* Workshop Quizzes (`WQ`)
* Hurdle Quizzes (`HQ`)

In [6]:
# CANVAS (upload requirement) keys
keys_CANVAS = [
    'LastName', 
    'FirstName', 
    'ID', 
    'SIS User ID', 
    'SIS Login ID',
    'Integration ID', 
    'Section',
]

# All workshop participation quizzes - some weeks might have multiple quizzes
keys_WQ = [
    'WQ04-01 Consumer Preferences (properties) (34473)',
    'WQ04-02 Preference Representation - ICs (34474)',
    'WQ04-03 Two-period Model: Walras (34475)',
    'WQ05-01 Firms and technology (34982)',
    'WQ06-01 Competitive Equilibrium (Static) (35456)',
    'WQ06-02 Static GE + distortionary labor income tax (35470)',
    'WQ07-01 Intertemporal consumption smoothing (36954)',
    'WQ07-02 Consumption smoothing and ICs (36955)',
    'WQ07-03 Government, Lending and Borrowing (36979)',
    'WQ07-04 Endowment Point (36980)',
    'WQ08-01 PAYG and Samuelson (38088)',
    'WQ09-01 Yd curve: Microfoundations (38664)',
]

# All hurdle quizzes
keys_HQ = [
    'PT04+HQ01 Static General Equilibrium (Hurdle Assessment) (33863)',
    'HQ02 (Week 6) Static General Equilibrium + Growth (10%) (30828)',
    'HQ03 (Week 8) Dynamic Choice + Timing of Taxation + Pensions (5%) (30831)',
    'HQ04 (Week 10) Intertemporal GE model with investment (10%) (30829)',
    'HQ05 (Week 12) Intertemporal monetary GE model + Special Topic (10%) (30830)',
]

# Ordered keys - ensure keys_CANVAS goes first, same as in CANVAS!
keys_quizzes = keys_HQ + keys_WQ
keys_select = keys_CANVAS + keys_quizzes
keys_select

['LastName',
 'FirstName',
 'ID',
 'SIS User ID',
 'SIS Login ID',
 'Integration ID',
 'Section',
 'PT04+HQ01 Static General Equilibrium (Hurdle Assessment) (33863)',
 'HQ02 (Week 6) Static General Equilibrium + Growth (10%) (30828)',
 'HQ03 (Week 8) Dynamic Choice + Timing of Taxation + Pensions (5%) (30831)',
 'HQ04 (Week 10) Intertemporal GE model with investment (10%) (30829)',
 'HQ05 (Week 12) Intertemporal monetary GE model + Special Topic (10%) (30830)',
 'WQ04-01 Consumer Preferences (properties) (34473)',
 'WQ04-02 Preference Representation - ICs (34474)',
 'WQ04-03 Two-period Model: Walras (34475)',
 'WQ05-01 Firms and technology (34982)',
 'WQ06-01 Competitive Equilibrium (Static) (35456)',
 'WQ06-02 Static GE + distortionary labor income tax (35470)',
 'WQ07-01 Intertemporal consumption smoothing (36954)',
 'WQ07-02 Consumption smoothing and ICs (36955)',
 'WQ07-03 Government, Lending and Borrowing (36979)',
 'WQ07-04 Endowment Point (36980)',
 'WQ08-01 PAYG and Samuelson (

In [7]:
df_hurdles = df[keys_select].fillna(0.0)
# df_hurdles 

In [8]:
# Ensure all values are normalized to 100% ...
# % row-0 in df_hurdles is max mark for each assessment item
dh_temp = (df_hurdles[keys_quizzes] * 100).div(df_hurdles[keys_quizzes].iloc[0], axis='columns')
# Concatenate CANVAS student keys with normalized dataframe of marks
dh = pd.concat([df_hurdles[keys_CANVAS], dh_temp], axis=1)
# dh.head(4)

## Workshop Quizzes Participation Hurdle (Task 2 in Course Summary)

* Starts: Week 4
* Ends: Week 12
* Freebies: Weeks 10, 11 and 12
* All WQ marks are recorded in percentages (i.e., out of 100%) and so are already normalized
* This section 
    * calculate averages for each week's activity, includes three final weeks of "freebies"
    * Sorts and picks out best 5 of 10 (see Course Summary, Task 2 assessment policy), so "reasonable misses" are not disadvantaged

In [9]:
keys_WQ4 = [
    'WQ04-01 Consumer Preferences (properties) (34473)',
    'WQ04-02 Preference Representation - ICs (34474)',
    'WQ04-03 Two-period Model: Walras (34475)',
]

keys_WQ5 = [
    'WQ05-01 Firms and technology (34982)',
]

keys_WQ6 = [
    'WQ06-01 Competitive Equilibrium (Static) (35456)',
    'WQ06-02 Static GE + distortionary labor income tax (35470)',
]

keys_WQ7 = [
    'WQ07-01 Intertemporal consumption smoothing (36954)',
    'WQ07-02 Consumption smoothing and ICs (36955)',
    'WQ07-03 Government, Lending and Borrowing (36979)',
    'WQ07-04 Endowment Point (36980)',
]

keys_WQ8 = [
    'WQ08-01 PAYG and Samuelson (38088)',
]

keys_WQ9 = [
    'WQ09-01 Yd curve: Microfoundations (38664)',
]

In [10]:
# NOTE: OFFICIAL ASSESSMENTS BEGIN IN WEEK 4, END WEEK 12
dh["WQ4-mean"] = dh[keys_WQ4].mean(axis=1)
dh["WQ5-mean"] = dh[keys_WQ5].mean(axis=1)
dh["WQ6-mean"] = dh[keys_WQ5].mean(axis=1)
dh["WQ7-mean"] = dh[keys_WQ5].mean(axis=1)
dh["WQ8-mean"] = dh[keys_WQ5].mean(axis=1)
dh["WQ9-mean"] = dh[keys_WQ5].mean(axis=1)
dh["WQ10-mean"] = 100.
dh["WQ11-mean"] = 100.
dh["WQ12-mean"] = 100.

In [11]:
# dh

In [12]:
cols = ["WQ4-mean", "WQ5-mean", "WQ6-mean",
        "WQ7-mean", "WQ8-mean", "WQ9-mean", "WQ10-mean", "WQ11-mean", "WQ12-mean"]

# Sort, and compute mean across Top 5 of 10 WQ items
dh["WQ-Top5-mean"] = dh[cols].fillna(0.0).apply(
    # Get the Top 4 values for each row
    lambda row: np.mean(row.nlargest(5).values), axis=1, result_type='expand'
)

In [13]:
# dh

## Calculate averages - Hurdle Quizzes (Task 1 in Course Summary)

There were 5 HQ assessments. Top 3 policy applies.

* Note late submissions are automatically 0.

* Some with insufficient submissions (less than 3).

This section computes the average of best 3 of 5 HQ assessment items.

In [ ]:
dh["HQ-Top3-mean"] = dh[keys_HQ].fillna(0).apply(
    # Get the Top 3 values for each row
    lambda row: np.mean(row.nlargest(3).values), axis=1, result_type='expand'
)

In [15]:
# dh

## Calculate course marks (%) for hurdles: Task 1 and Task 2

Given the calculations from the last two sections, this section computes the weighted average
with weights specified by Course Summary as:

* Task 1 - HQ (40%)
* Task 2 - WQ (20%)

As mandated in the definition of hurdle assessments in [ANU Policy: Student Assessment (coursework), section 12](https://policies.anu.edu.au/ppl/document/ANUP_004603#:~:text=of%20this%20policy.-,Hurdle,-assessments%20are%20identified), a fail in Task 1 *and* Task 2 constitutes a fail in this course. 

* Implementation-wise, this section sets the weighted sum to zero if this sum is strictly less than 30 marks (i.e., a fail mark). (60 marks is the maximum.)

In [16]:
dh["All Hurdles (Max 60)"] = 0.2*dh["WQ-Top5-mean"] + 0.4*dh["HQ-Top3-mean"]

# If less than 30, All Hurdles == 0 mark.
dh.loc[dh["All Hurdles (Max 60)"] < 30.0, "All Hurdles (Max 60)"] = 0.0

## Export results back to `CANVAS`

In [17]:
# Export your DataFrame, substitute empty spaces for NaN values
dh.round(0).to_csv(official_outfile, na_rep='', index=False)

In [18]:
# Hurdles - dataframe, summary
key_dhs = keys_CANVAS  \
        + ["WQ-Top5-mean", "HQ-Top3-mean", "All Hurdles (Max 60)"]
dhs = dh[key_dhs]

In [19]:
# dhs

In [20]:
hurds = dhs["All Hurdles (Max 60)"]

### Summary statistics

In [21]:
moments = pd.DataFrame(hurds.apply(stat_au_choix))
moments.round(0)

,All Hurdles (Max 60)
mean,53.0
median,59.0
std,16.0
min,0.0
max,60.0
len,94.0


In [22]:
moments.round(0).to_html(officialpath + "moments.html",
                         classes='table table-striped table-hover table-bordered',
                         )

In [23]:
deciles = np.arange(0.1, 1.0, 0.1).tolist()
pct = (hurds.quantile(deciles).to_frame().rename_axis("Percentile (decimals)")).round(0)
pct

,All Hurdles (Max 60)
Percentile (decimals),
0.1,47.0
0.2,54.0
0.3,56.0
0.4,58.0
0.5,59.0
0.6,59.0
0.7,60.0
0.8,60.0
0.9,60.0


In [24]:
pct.to_html(officialpath + "percentiles.html",
            classes='table table-striped table-hover table-bordered',
            )